# NOUS — Neural Omnidimensional Unified System
**Computation as physical relaxation. No BPTT. No attention.**

| Step | What it does |
|------|--------------|
| Cell 1 | GPU check |
| Cell 2 | Mount Drive (checkpoints persist) |
| Cell 3 | Install deps + clone repo |
| Cell 4 | Train (self-contained, just run this) |

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────
import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/nous_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints → {CKPT_DIR}')

In [ ]:
# ── Cell 3: Install + clone ───────────────────────────────────────────────
!pip install -q transformers datasets
!git clone https://github.com/HemanthKiranPolu/nous.git /content/nous 2>/dev/null || (cd /content/nous && git pull)
!cd /content/nous && git checkout scale/nous-7b && git log --oneline -2

In [ ]:
# ── Cell 4: NOUS Training (self-contained) ────────────────────────────────
#
# Tune these:
ODE_STEPS  = 10          # 10 = fast/approximate | 80 = slow/accurate
BATCH_SIZE = 4           # T4: 4 | V100: 8 | A100: 16
MAX_TOKENS = 5_000_000   # 5M = quick run | 100M = overnight
MAX_LEN    = 16          # max tokens per sentence
DATASET    = 'wikitext103'  # 'wikitext103' | 'c4'
# ─────────────────────────────────────────────────────────────────────────

import sys, os, math, time, glob
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import replace
from transformers import GPT2Tokenizer
from datasets import load_dataset

sys.path.insert(0, '/content/nous')
from nous.nous7b.config import NOUS_SMALL
from nous.nous7b.energy_net_7b import NOUSEnergyNet7B
from nous.nous7b.ode_batched import BatchedEulerODE

CKPT_DIR = '/content/drive/MyDrive/nous_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

device    = torch.device('cuda')
dtype     = torch.float32
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
cfg = replace(NOUS_SMALL,
              vocab_size=tokenizer.vocab_size,
              n_steps_free=ODE_STEPS,
              n_steps_nudge=ODE_STEPS,
              eps=0.3)

model = NOUSEnergyNet7B(cfg).to(device=device, dtype=dtype)
opt   = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                           weight_decay=0.1, betas=(0.9, 0.95))
ode   = BatchedEulerODE(model, cfg)
model.parameter_summary()

# ── Resume from checkpoint ────────────────────────────────────────────────
start_tokens = 0
ckpts = sorted(glob.glob(f'{CKPT_DIR}/step_*.pt'))
if ckpts:
    ck = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    start_tokens = ck.get('total_tokens', 0)
    print(f'Resumed from {ckpts[-1]}  ({start_tokens/1e6:.1f}M tokens)')
else:
    print('Starting fresh')

# ── Data stream ───────────────────────────────────────────────────────────
def sentence_stream(dataset_name, tokenizer, min_len=4, max_len=MAX_LEN):
    if dataset_name == 'wikitext103':
        ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train')
    elif dataset_name == 'c4':
        ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    else:
        ds = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
    for item in ds:
        t = item['text'].strip()
        if not t or t.startswith('='): continue
        ids = tokenizer.encode(t)
        if min_len <= len(ids) <= max_len:
            yield ids

def batch_stream(stream, B):
    buf = []
    for s in stream:
        buf.append(s)
        if len(buf) == B:
            yield buf; buf = []
    if buf: yield buf

def param_grad(model, x_embed, q_star):
    for p in model.parameters():
        if p.grad is not None: p.grad.zero_()
    E = model(x_embed, q_star.detach())
    E.backward()
    return {n: p.grad.clone() if p.grad is not None else torch.zeros_like(p)
            for n, p in model.named_parameters()}

# ── Training loop ─────────────────────────────────────────────────────────
stream       = batch_stream(sentence_stream(DATASET, tokenizer), BATCH_SIZE)
global_step  = 0
total_tokens = start_tokens
loss_hist    = []
t0           = time.time()

print(f'\n{"Step":>6}  {"PPL":>8}  {"Mtok":>6}  {"tok/s":>6}')
print('─' * 36)

for batch_idx, batch_sents in enumerate(stream):
    B_cur    = len(batch_sents)
    q_states = torch.zeros(B_cur, cfg.state_dim, device=device, dtype=dtype)
    max_t    = max(len(s) for s in batch_sents) - 1
    lens     = [len(s) - 1 for s in batch_sents]

    opt.zero_grad()

    for t in range(max_t):
        active = [b for b in range(B_cur) if t < lens[b]]
        if not active: continue

        x_t = torch.stack([
            model.embedding(torch.tensor(batch_sents[b][t], device=device)).detach()
            for b in active
        ]).to(dtype)
        tgt_t = torch.tensor([batch_sents[b][t+1] for b in active], device=device)
        q0_t  = q_states[active]

        # Free phase
        q_free = ode.solve_batch(x_t, q0_t, n_steps=cfg.n_steps_free)

        # Nudged phase
        extra = [(lambda i: lambda q: cfg.eps * F.cross_entropy(
                      model.decode(q).unsqueeze(0), tgt_t[i:i+1]))(i)
                 for i in range(len(active))]
        q_nudge = ode.solve_batch(x_t, q0_t, n_steps=cfg.n_steps_nudge, extra_fns=extra)

        # EqProp gradient accumulation
        scale = 1.0 / (len(active) * cfg.eps)
        step_loss = 0.0
        for i, b in enumerate(active):
            gf = param_grad(model, x_t[i], q_free[i])
            gn = param_grad(model, x_t[i], q_nudge[i])
            for name, param in model.named_parameters():
                if param.requires_grad and name in gf:
                    g = scale * (gn[name] - gf[name])
                    param.grad = g if param.grad is None else param.grad.add_(g)
            with torch.no_grad():
                step_loss += F.cross_entropy(
                    model.decode(q_free[i]).unsqueeze(0), tgt_t[i:i+1]).item()
            q_states[b] = q_free[i]

        # Decoder CE on nudged equilibrium
        ce = sum(F.cross_entropy(model.decode(q_nudge[i]).unsqueeze(0), tgt_t[i:i+1])
                 for i in range(len(active))) / len(active)
        ce.backward()
        loss_hist.append(step_loss / len(active))
        total_tokens += len(active)

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    global_step += 1

    if global_step % 10 == 0:
        avg = np.mean(loss_hist[-100:]) if loss_hist else float('nan')
        ppl = math.exp(min(avg, 20))
        tps = (total_tokens - start_tokens) / max(time.time() - t0, 1)
        print(f'{global_step:6d}  {ppl:8.1f}  {total_tokens/1e6:6.3f}  {tps:6.0f}', flush=True)

    if global_step % 500 == 0:
        ckpt_path = f'{CKPT_DIR}/step_{global_step:07d}.pt'
        torch.save({'step': global_step, 'total_tokens': total_tokens,
                    'model': model.state_dict(), 'opt': opt.state_dict(),
                    'loss': float(np.mean(loss_hist[-200:]))}, ckpt_path)
        print(f'  → saved {ckpt_path}')

    if total_tokens - start_tokens >= MAX_TOKENS:
        print(f'Done — {(total_tokens-start_tokens)/1e6:.1f}M tokens.')
        break

In [ ]:
# ── Cell 5: Plot PPL curve ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np, math

w = 200
ppls = [math.exp(min(np.mean(loss_hist[max(0,i-w):i+1]), 20))
        for i in range(0, len(loss_hist), 20)]

plt.figure(figsize=(10, 4))
plt.plot(ppls, color='#9060ff', lw=2)
plt.xlabel('Steps'); plt.ylabel('Perplexity'); plt.yscale('log')
plt.title(f'NOUS-Small — WikiText-103 — {total_tokens/1e6:.1f}M tokens')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final PPL: {ppls[-1]:.1f}')

In [ ]:
# ── Cell 6: Greedy inference ──────────────────────────────────────────────
model.eval()
prompt   = 'The neural network learned'
ids      = tokenizer.encode(prompt)
q        = torch.zeros(cfg.state_dim, device=device, dtype=dtype)

for tok_id in ids[:-1]:
    x = model.embedding(torch.tensor(tok_id, device=device)).detach()
    q = ode.solve_batch(x.unsqueeze(0), q.unsqueeze(0),
                        n_steps=cfg.n_steps_free)[0]

generated = list(ids)
for _ in range(40):
    x = model.embedding(torch.tensor(generated[-1], device=device)).detach()
    q = ode.solve_batch(x.unsqueeze(0), q.unsqueeze(0),
                        n_steps=cfg.n_steps_free)[0]
    next_tok = model.decode(q).argmax().item()
    generated.append(next_tok)
    if next_tok == tokenizer.eos_token_id: break

print('Output:', tokenizer.decode(generated))